# Data Extraction & Web Scraping — Self-Study Guide

Welcome! This notebook teaches you how to collect data from the internet using three essential techniques:

1. **REST API** — request structured data from a server using standard HTTP calls
2. **GraphQL** — ask for exactly the fields you need, nothing more
3. **Web Scraping** — extract data directly from HTML when no API exists

**By the end of this notebook you will be able to:**
- Authenticate and paginate through a real REST API (GitHub)
- Write GraphQL queries to fetch precise data
- Parse HTML with BeautifulSoup and handle multi-page scrapes

> **Who this is for:** You know basic Python (variables, loops, functions). You've never called an API or scraped a website before. No worries — everything is explained step by step.

## What You'll Build

By the end of this notebook you'll have extracted five real datasets:

| # | Dataset | How | Output |
|---|---------|-----|--------|
| 1 | `pandas` library release history | REST API | `pandas_releases` DataFrame + `pandas_releases` DuckDB table |
| 2 | Top contributors by commit count | REST API | `contributors` DataFrame + `pandas_contributors` DuckDB table |
| 3 | GraphQL query of `pandas` releases | GraphQL | `graphql_releases` DataFrame |
| 4 | Countries of the world (name, capital, population, area) | Web scraping | `countries_df` DataFrame |
| 5 | NHL hockey team stats (all 24 pages) | Web scraping + pagination | `hockey_stats` DataFrame |

Here's a sneak peek at what `pandas_releases` will look like:

```
   version          published_at                          summary
0  v2.2.1   2024-03-18 15:30:00+00:00  ## What's new in 2.2.1...
1  v2.2.0   2024-01-19 17:00:00+00:00  ## What's new in 2.2.0...
2  v2.1.4   2024-01-08 20:00:00+00:00  ## What's new in 2.1.4...
...
```

---
## Setup

Before we write any code, let's get our tools ready.

### Libraries we'll use

| Library | What it does |
|---------|-------------|
| `requests` | Makes HTTP calls to APIs and websites |
| `python-dotenv` | Loads secret keys from a `.env` file so they stay out of your code |
| `pandas` | Creates DataFrames — like spreadsheets in Python |
| `sqlalchemy` | Lets pandas talk to databases |
| `duckdb` | A fast, file-based database — we'll save our results here |
| `beautifulsoup4` | Parses HTML so we can extract data from web pages |

Install them if needed:
```bash
pip install requests python-dotenv pandas sqlalchemy duckdb duckdb-engine beautifulsoup4
```

### Step 1 of 2: Get a GitHub Personal Access Token

We need a token so GitHub knows who we are and lets us call their API.

**Follow these steps:**

1. Go to [github.com](https://github.com) and sign in
2. Click your profile photo (top-right) → **Settings**
3. Scroll down the left sidebar → click **Developer settings** (at the very bottom)
4. Click **Personal access tokens** → **Fine-grained tokens** → **Generate new token**
5. Give it a name (e.g. `self-study-scraping`)
6. Set **Expiration** to 30 days
7. Under **Repository access**, select **Public repositories (read-only)**
8. Scroll to the bottom → click **Generate token**
9. **Copy the token immediately** — GitHub only shows it once!

> ⚠️ Keep your token private. Never paste it directly into a notebook or commit it to GitHub.

In [ ]:
import os

# Run this cell ONCE to create your .env file.
# After running, open the .env file and paste your token in place of the placeholder.

env_path = '../.env'

if os.path.exists(env_path):
    print(f"ℹ️  .env file already exists at '{env_path}'.")
    print("   Open it and make sure GITHUB_TOKEN is set to your actual token.")
else:
    with open(env_path, 'w') as f:
        f.write("# GitHub personal access token\n")
        f.write("GITHUB_TOKEN='<PASTE-YOUR-TOKEN-HERE>'\n")
    print(f"✅ Created '{env_path}'")
    print("   Now open the file and replace <PASTE-YOUR-TOKEN-HERE> with your actual token.")

### Step 2 of 2: Add your token to the .env file

1. Find the `.env` file in the **root folder** of this project (one level above `notebooks/`)
2. Open it in a text editor
3. Replace `<PASTE-YOUR-TOKEN-HERE>` with your actual token, like this:

```
GITHUB_TOKEN='ghp_xxxxxxxxxxxxxxxxxxxx'
```

4. Save and close the file

> 📁 The `.env` file is listed in `.gitignore` so it will **never** be pushed to GitHub.

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()  # 👈 reads the .env file and loads GITHUB_TOKEN into the environment
access_token = os.getenv('GITHUB_TOKEN')

# Verify the token loaded correctly (prints first/last 4 chars only — never print the full token!)
if access_token and access_token != '<PASTE-YOUR-TOKEN-HERE>':
    if len(access_token) >= 8:
        print(f"✅ Token loaded: {access_token[:4]}...{access_token[-4:]}")
    else:
        print("⚠️  Token looks too short — double-check you copied it in full.")
else:
    print("❌ Token not found or still a placeholder. Check your .env file.")

---
## Chapter 1 — REST API

### The Mission

You've joined a team that tracks the health of popular open-source libraries. Your first task: **collect the full release history of the `pandas` library** — every version, when it was published, and what changed.

Pandas hosts all of this on GitHub. GitHub exposes it via a **REST API** — a web service that returns structured data when you send it the right request. Let's learn how.

### Concept: How a REST API Works

When you call an API, your Python script sends an **HTTP request** to a server. The server processes it and sends back an **HTTP response** containing data (usually in JSON format).

<svg viewBox="0 0 620 110" xmlns="http://www.w3.org/2000/svg" style="font-family:sans-serif;font-size:12px;max-width:620px;display:block;margin:16px 0;">
  <rect x="10" y="35" width="120" height="42" rx="6" fill="#dbeafe" stroke="#3b82f6" stroke-width="1.5"/>
  <text x="70" y="57" text-anchor="middle" fill="#1e40af" font-weight="bold">Your Script</text>
  <text x="70" y="71" text-anchor="middle" fill="#1e40af" font-size="10">(requests.get)</text>
  <rect x="490" y="35" width="120" height="42" rx="6" fill="#dcfce7" stroke="#22c55e" stroke-width="1.5"/>
  <text x="550" y="57" text-anchor="middle" fill="#166534" font-weight="bold">GitHub API</text>
  <text x="550" y="71" text-anchor="middle" fill="#166534" font-size="10">api.github.com</text>
  <defs>
    <marker id="arr-blue" markerWidth="8" markerHeight="6" refX="7" refY="3" orient="auto"><polygon points="0 0,8 3,0 6" fill="#3b82f6"/></marker>
    <marker id="arr-green" markerWidth="8" markerHeight="6" refX="7" refY="3" orient="auto"><polygon points="0 0,8 3,0 6" fill="#22c55e"/></marker>
  </defs>
  <line x1="130" y1="50" x2="488" y2="50" stroke="#3b82f6" stroke-width="1.5" marker-end="url(#arr-blue)"/>
  <text x="310" y="44" text-anchor="middle" fill="#1d4ed8" font-size="11">① GET /repos/pandas-dev/pandas/releases   +   Authorization header</text>
  <line x1="490" y1="63" x2="132" y2="63" stroke="#22c55e" stroke-width="1.5" marker-end="url(#arr-green)"/>
  <text x="310" y="80" text-anchor="middle" fill="#15803d" font-size="11">② 200 OK   +   JSON array of release objects</text>
</svg>

**Key terms:**

| Term | What it means |
|------|---------------|
| **Endpoint** | The URL you call, e.g. `https://api.github.com/repos/pandas-dev/pandas/releases` |
| **HTTP verb** | The type of action — `GET` = read data, `POST` = send data |
| **Header** | Extra info sent with the request — e.g. your identity token |
| **Status code** | A 3-digit number in the response: `200` = success, `4xx` = your error, `5xx` = server error |
| **JSON** | The data format APIs return — looks like a Python dict/list |
| **Bearer token** | Your API key, passed in the `Authorization` header |

### Concept: JSON is just Python dicts and lists

JSON responses from APIs look like this:

```json
[
  {
    "tag_name": "v2.2.1",
    "published_at": "2024-03-18T15:30:00Z",
    "body": "## What's new in 2.2.1 ..."
  },
  {
    "tag_name": "v2.2.0",
    "published_at": "2024-01-19T17:00:00Z",
    "body": "## What's new in 2.2.0 ..."
  }
]
```

- The outer `[...]` means it's a **list**
- Each `{...}` is a **dict** with key-value pairs
- In Python: `response.json()` gives you the list. `response.json()[0]["tag_name"]` gives you `"v2.2.1"`